# eBay Deal Finder (Python Notebook)

Notebook-first implementation for finding underpriced eBay listings via active-price distribution, sold-volume enrichment, and optional Azure OpenAI listing evaluation.


## 1) Imports and Configuration


In [10]:
import os
import json
import base64
import math
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

import pandas as pd
from IPython.display import display, HTML
import requests
from dotenv import load_dotenv
from openai import (
    AzureOpenAI,
    APIConnectionError,
    APIStatusError,
    APITimeoutError,
    RateLimitError,
)

load_dotenv(override=True)

def require_env(name):
    value = os.environ.get(name, "").strip()
    if not value:
        raise ValueError(f"{name} must be set in .env")
    return value

EBAY_APP_ID = require_env("EBAY_APP_ID")
EBAY_CERT_ID = require_env("EBAY_CERT_ID")
EBAY_SANDBOX = os.environ.get("EBAY_SANDBOX", "false").lower() == "true"
EBAY_MARKETPLACE_ID = os.environ.get("EBAY_MARKETPLACE_ID", "EBAY_US")

AZURE_OPENAI_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT", "").strip()
AZURE_OPENAI_API_KEY = os.environ.get("AZURE_OPENAI_API_KEY", "").strip()
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "").strip()
LLM_ENABLED = bool(AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY and AZURE_OPENAI_DEPLOYMENT)

SEARCH_QUERY = '"TI-84 Plus CE" graphing calculator'
SEARCH_EXCLUSIONS = '-charger -case -cable -cover -emulator -software -cord -bag -pouch -dock -screen -protector -"parts only" -"for parts" -stand -holder -skin -sticker -keypad'
CONDITION = "Used"
MIN_Z_SCORE = -1.0
EBAY_FEE_RATE = 0.13
RESULTS_PER_PAGE = 200
MAX_LLM_CALLS = 200  # Keep all statistical candidates unless budget is explicitly lower
MAX_IMAGES_PER_LISTING = 5
MIN_MARKET_SAMPLE = 20
MIN_STD_DEV = 0.01
FALLBACK_DISCOUNT_RATE = 0.10
API_TIMEOUT_S = 15
LLM_TIMEOUT_S = 45
API_MAX_RETRIES = 2
API_RETRY_BASE_DELAY_S = 0.5
LLM_CONCURRENCY = 5

BUYER_COUNTRY = os.environ.get("BUYER_COUNTRY", "US").strip()
BUYER_POSTAL_CODE = os.environ.get("BUYER_POSTAL_CODE", "").strip()

EBAY_API_BASE = "https://api.sandbox.ebay.com" if EBAY_SANDBOX else "https://api.ebay.com"
EBAY_AUTH_URL = f"{EBAY_API_BASE}/identity/v1/oauth2/token"
EBAY_BROWSE_URL = f"{EBAY_API_BASE}/buy/browse/v1"

print(f"Environment:  {'SANDBOX' if EBAY_SANDBOX else 'PRODUCTION'}")
print(f"Marketplace:  {EBAY_MARKETPLACE_ID}")
print(f"LLM enabled:  {LLM_ENABLED}")
print(f"Buyer location: {BUYER_COUNTRY} {BUYER_POSTAL_CODE or '(no postal code)'}")
print(f"Search query: {SEARCH_QUERY}")


Environment:  PRODUCTION
Marketplace:  EBAY_US
LLM enabled:  True
Buyer location: US 07675
Search query: "TI-84 Plus CE" graphing calculator


## 2) Shared Helpers (Stats, Retry, Ranking)


In [11]:
RETRYABLE_EXCEPTIONS = (
    requests.exceptions.RequestException,
    APITimeoutError,
    APIConnectionError,
    RateLimitError,
    APIStatusError,
)

def format_currency(n):
    return f"${n:.2f}"

def average(numbers):
    return sum(numbers) / len(numbers)

def median(numbers):
    sorted_numbers = sorted(numbers)
    mid = len(sorted_numbers) // 2
    if len(sorted_numbers) % 2 == 0:
        return (sorted_numbers[mid - 1] + sorted_numbers[mid]) / 2
    return sorted_numbers[mid]

def std_dev(numbers, mean):
    squared_diffs = [(n - mean) ** 2 for n in numbers]
    return math.sqrt(sum(squared_diffs) / len(numbers))

def filter_outliers(prices):
    sorted_prices = sorted(prices)
    q1 = sorted_prices[int(len(sorted_prices) * 0.25)]
    q3 = sorted_prices[int(len(sorted_prices) * 0.75)]
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    filtered = [p for p in sorted_prices if lower <= p <= upper]
    return {
        "filtered": filtered,
        "lower": max(lower, 0),
        "upper": upper,
        "removed_count": len(prices) - len(filtered),
    }

def normalize_condition(condition):
    return re.sub(r"\s+", "_", condition.strip().upper())

def extract_status_code(exc):
    if isinstance(exc, requests.exceptions.HTTPError) and exc.response is not None:
        return exc.response.status_code

    for attr in ("status_code", "status", "statusCode"):
        status = getattr(exc, attr, None)
        if isinstance(status, int):
            return status

    response = getattr(exc, "response", None)
    if response is not None:
        status = getattr(response, "status_code", None)
        if isinstance(status, int):
            return status

    return None

def is_retryable_error(exc):
    if isinstance(
        exc,
        (
            requests.exceptions.Timeout,
            requests.exceptions.ConnectionError,
            APITimeoutError,
            APIConnectionError,
            RateLimitError,
        ),
    ):
        return True

    status = extract_status_code(exc)
    if isinstance(status, int):
        return status in (408, 429) or status >= 500

    msg = str(exc).lower()
    return any(token in msg for token in ("timeout", "timed out", "network", "connection reset", "socket"))

def call_with_retry(label, fn, timeout_s, max_retries=API_MAX_RETRIES, base_delay_s=API_RETRY_BASE_DELAY_S):
    for attempt in range(1, max_retries + 2):
        try:
            return fn(timeout_s)
        except RETRYABLE_EXCEPTIONS as exc:
            should_retry = attempt <= max_retries and is_retryable_error(exc)
            if not should_retry:
                raise

            wait_s = base_delay_s * (2 ** (attempt - 1))
            print(
                f"⚠ {label} failed (attempt {attempt}/{max_retries + 1}): {exc}. "
                f"Retrying in {wait_s:.1f}s..."
            )
            time.sleep(wait_s)

def extract_total_cost(item):
    """Extract price, shipping cost, and total cost from an eBay item dict.
    
    Works with both itemSummary (search) and full item (getItem) responses.
    Returns dict with price, shipping_cost, total_cost, and shipping_type.
    """
    try:
        price = float(item.get("price", {}).get("value", 0))
    except (TypeError, ValueError):
        price = 0.0

    shipping_cost = 0.0
    shipping_type = "FREE"
    shipping_options = item.get("shippingOptions") or []

    if shipping_options and isinstance(shipping_options[0], dict):
        option = shipping_options[0]
        cost_obj = option.get("shippingCost") or {}
        cost_type = option.get("shippingCostType", "")

        try:
            shipping_cost = float(cost_obj.get("value", 0))
        except (TypeError, ValueError):
            shipping_cost = 0.0

        if shipping_cost > 0:
            shipping_type = cost_type or "FIXED"
        elif cost_type == "CALCULATED" and shipping_cost == 0:
            # API returned 0 for calculated — might be free or might be unknown
            shipping_type = "CALCULATED"
        else:
            shipping_type = "FREE"
    else:
        shipping_type = "UNKNOWN"

    return {
        "price": price,
        "shipping_cost": shipping_cost,
        "total_cost": price + shipping_cost,
        "shipping_type": shipping_type,
    }

def price_rank_from_signal(signal):
    if "Strong" in signal:
        return 3
    if "Buy" in signal:
        return 2
    return 1

def price_only_recommendation(price_rank):
    if price_rank >= 3:
        return "✅ BUY"
    if price_rank >= 2:
        return "⚠️  CONSIDER"
    return "❌ SKIP"

print("✅ Shared helpers loaded")


✅ Shared helpers loaded


## 3) eBay API Client and Shared Item Cache


In [12]:
_EBAY_ACCESS_TOKEN = None
_EBAY_TOKEN_EXPIRES_AT = 0
ITEM_DETAILS_CACHE = {}

def get_ebay_token(timeout_s):
    credentials = base64.b64encode(f"{EBAY_APP_ID}:{EBAY_CERT_ID}".encode()).decode()

    def _request(request_timeout_s):
        response = requests.post(
            EBAY_AUTH_URL,
            headers={
                "Content-Type": "application/x-www-form-urlencoded",
                "Authorization": f"Basic {credentials}",
            },
            data={
                "grant_type": "client_credentials",
                "scope": "https://api.ebay.com/oauth/api_scope",
            },
            timeout=request_timeout_s,
        )
        response.raise_for_status()
        return response.json()

    token_payload = call_with_retry("eBay OAuth token", _request, timeout_s)
    return token_payload["access_token"], int(token_payload.get("expires_in", 0))

def get_ebay_headers(timeout_s=API_TIMEOUT_S):
    global _EBAY_ACCESS_TOKEN
    global _EBAY_TOKEN_EXPIRES_AT

    now = time.time()
    if not (_EBAY_ACCESS_TOKEN and now < _EBAY_TOKEN_EXPIRES_AT - 60):
        token, expires_in = get_ebay_token(timeout_s)
        _EBAY_ACCESS_TOKEN = token
        _EBAY_TOKEN_EXPIRES_AT = now + expires_in

    headers = {
        "Authorization": f"Bearer {_EBAY_ACCESS_TOKEN}",
        "X-EBAY-C-MARKETPLACE-ID": EBAY_MARKETPLACE_ID,
        "Content-Type": "application/json",
    }
    if BUYER_POSTAL_CODE:
        headers["X-EBAY-C-ENDUSERCTX"] = (
            f"contextualLocation=country%3D{BUYER_COUNTRY}%2Czip%3D{BUYER_POSTAL_CODE}"
        )
    return headers

def browse_search(params, label):
    def _request(request_timeout_s):
        response = requests.get(
            f"{EBAY_BROWSE_URL}/item_summary/search",
            headers=get_ebay_headers(request_timeout_s),
            params=params,
            timeout=request_timeout_s,
        )
        response.raise_for_status()
        return response.json()

    return call_with_retry(label, _request, API_TIMEOUT_S)

def browse_get_item(item_id, label):
    def _request(request_timeout_s):
        response = requests.get(
            f"{EBAY_BROWSE_URL}/item/{item_id}",
            headers=get_ebay_headers(request_timeout_s),
            timeout=request_timeout_s,
        )
        response.raise_for_status()
        return response.json()

    return call_with_retry(label, _request, API_TIMEOUT_S)

def clear_item_details_cache():
    ITEM_DETAILS_CACHE.clear()

def get_item_details(item_id):
    if item_id in ITEM_DETAILS_CACHE:
        return ITEM_DETAILS_CACHE[item_id]

    item_details = browse_get_item(item_id, f"Browse getItem for {item_id}")
    ITEM_DETAILS_CACHE[item_id] = item_details
    return item_details

print("✅ API helpers and item cache loaded")


✅ API helpers and item cache loaded


## 4) Market Sampling and Pricing Analysis


In [13]:
def get_market_listings(query, condition):
    full_query = f"{query} {SEARCH_EXCLUSIONS}" if SEARCH_EXCLUSIONS else query
    condition_filter = normalize_condition(condition)
    all_items = []
    max_items = 10_000
    page = 0

    print(f"\n📊 Fetching active market listings for '{query}' ({condition})...\n")

    while len(all_items) < max_items:
        params = {
            "q": full_query,
            "filter": f"buyingOptions:{{FIXED_PRICE}},conditions:{{{condition_filter}}},priceCurrency:USD",
            "limit": str(RESULTS_PER_PAGE),
            "offset": str(page * RESULTS_PER_PAGE),
        }

        try:
            result = browse_search(params, f"Browse market listings page {page + 1}")
        except RETRYABLE_EXCEPTIONS as exc:
            if not all_items:
                raise RuntimeError(f"Unable to fetch market listings page {page + 1}: {exc}") from exc
            print(
                f"⚠ Market listing fetch stopped at page {page + 1}: {exc}. "
                f"Continuing with {len(all_items)} result(s)."
            )
            break

        items = result.get("itemSummaries") or []
        if not items:
            break

        all_items.extend(items)
        total = result.get("total", 0)
        if len(all_items) >= total:
            break

        page += 1

    return all_items

def analyze_prices(market_items):
    total_costs = []
    shipping_costs = []
    for item in market_items:
        cost_info = extract_total_cost(item)
        if cost_info["price"] > 0:
            total_costs.append(cost_info["total_cost"])
            shipping_costs.append(cost_info["shipping_cost"])

    if not total_costs:
        print("❌ No market price data found.")
        return None
    if len(total_costs) < MIN_MARKET_SAMPLE:
        print(
            f"❌ Not enough market listings for reliable pricing "
            f"({len(total_costs)} found, need at least {MIN_MARKET_SAMPLE})."
        )
        return None

    avg_shipping = average(shipping_costs)
    med_shipping = median(shipping_costs)
    sorted_costs = sorted(total_costs)
    print("─" * 50)
    print("  RAW MARKET TOTAL COSTS (price + shipping)")
    print("─" * 50)
    print(f"  Items analyzed:    {len(total_costs)}")
    print(f"  Average total:     {format_currency(average(total_costs))}")
    print(f"  Median total:      {format_currency(median(total_costs))}")
    print(f"  Low / High:        {format_currency(sorted_costs[0])} — {format_currency(sorted_costs[-1])}")
    print(f"  Avg shipping:      {format_currency(avg_shipping)}")
    print(f"  Median shipping:   {format_currency(med_shipping)}")
    print("─" * 50)

    outlier_result = filter_outliers(total_costs)
    filtered = outlier_result["filtered"]
    if not filtered:
        print("❌ All total costs were filtered as outliers. Check your data.")
        return None
    if len(filtered) < MIN_MARKET_SAMPLE:
        print(
            f"❌ Too few listings after outlier filtering "
            f"({len(filtered)} kept, need at least {MIN_MARKET_SAMPLE})."
        )
        return None

    filtered_sorted = sorted(filtered)
    filtered_mean = average(filtered)
    filtered_median = median(filtered)
    filtered_std_raw = std_dev(filtered, filtered_mean)
    has_usable_std_dev = math.isfinite(filtered_std_raw) and filtered_std_raw >= MIN_STD_DEV
    filtered_std = filtered_std_raw if has_usable_std_dev else 0
    cv = (filtered_std / filtered_mean) if has_usable_std_dev else 0

    stats = {
        "count": len(filtered),
        "average": filtered_mean,
        "median": filtered_median,
        "std_dev": filtered_std,
        "has_usable_std_dev": has_usable_std_dev,
        "cv": cv,
        "low": filtered_sorted[0],
        "high": filtered_sorted[-1],
        "p25": filtered_sorted[int(len(filtered_sorted) * 0.25)],
        "p75": filtered_sorted[int(len(filtered_sorted) * 0.75)],
        "median_shipping": med_shipping,
        "average_shipping": avg_shipping,
    }

    print(f"\n  🧹 IQR FILTER: keeping {format_currency(outlier_result['lower'])} — {format_currency(outlier_result['upper'])}")
    print(f"     Removed {outlier_result['removed_count']} outlier(s)\n")
    print("─" * 50)
    print("  FILTERED MARKET VALUE (total cost = price + shipping)")
    print("─" * 50)
    print(f"  Items kept:      {stats['count']} of {len(total_costs)}")
    print(f"  Average total:   {format_currency(stats['average'])}")
    print(f"  Median total:    {format_currency(stats['median'])}")
    print(f"  Std deviation:   {format_currency(stats['std_dev'])}")
    print(f"  Low / High:      {format_currency(stats['low'])} — {format_currency(stats['high'])}")
    print(f"  25th / 75th pct: {format_currency(stats['p25'])} — {format_currency(stats['p75'])}")
    print(f"  Median shipping: {format_currency(stats['median_shipping'])}")
    print("─" * 50)

    print("\n  📐 TOTAL COST BANDS")
    print("─" * 50)
    if stats["has_usable_std_dev"]:
        print(f"  -2σ (strong buy): {format_currency(stats['median'] - 2 * stats['std_dev'])}")
        print(f"  -1σ (buy):        {format_currency(stats['median'] - 1 * stats['std_dev'])}")
        print(f"   0  (fair value): {format_currency(stats['median'])}")
        print(f"  +1σ (overpriced): {format_currency(stats['median'] + 1 * stats['std_dev'])}")
        print(f"  +2σ (avoid):      {format_currency(stats['median'] + 2 * stats['std_dev'])}")
    else:
        fallback_cost = stats['median'] * (1 - FALLBACK_DISCOUNT_RATE)
        print(
            f"  ⚠ Std deviation is below {MIN_STD_DEV}; using discount scoring instead of z-score."
        )
        print(
            f"  Discount threshold ({(FALLBACK_DISCOUNT_RATE * 100):.0f}% below median): {format_currency(fallback_cost)}"
        )
    print("─" * 50)

    if not stats["has_usable_std_dev"]:
        market_verdict = "⚠️  Very low variance — using discount-based scoring"
    elif stats["cv"] < 0.3:
        market_verdict = "✅ Tight market — good for arbitrage"
    elif stats["cv"] < 0.5:
        market_verdict = "⚠️  Moderate spread — proceed with caution"
    else:
        market_verdict = "❌ High variance — no reliable fair value"

    cv_display = f"{stats['cv']:.3f}" if stats["has_usable_std_dev"] else "N/A"
    print(f"\n  CV: {cv_display}  →  {market_verdict}")
    print("─" * 50)

    return stats

print("✅ Market analysis functions loaded")


✅ Market analysis functions loaded


## 5) Deal Discovery and Sold-Quantity Enrichment


In [14]:
def find_deal_listings(query, condition, market_stats):
    has_usable_std_dev = market_stats["has_usable_std_dev"]
    raw_max_total = (
        market_stats["median"] + MIN_Z_SCORE * market_stats["std_dev"]
        if has_usable_std_dev
        else market_stats["median"] * (1 - FALLBACK_DISCOUNT_RATE)
    )
    max_total_cost = max(1, math.floor(raw_max_total))
    full_query = f"{query} {SEARCH_EXCLUSIONS}" if SEARCH_EXCLUSIONS else query
    condition_filter = normalize_condition(condition)
    all_items = []
    page = 0
    max_items = 10_000

    threshold_message = (
        f"(z \u2264 {MIN_Z_SCORE}, i.e. {abs(MIN_Z_SCORE)}\u03c3 below {format_currency(market_stats['median'])} median total cost)"
        if has_usable_std_dev
        else f"({(FALLBACK_DISCOUNT_RATE * 100):.0f}% below {format_currency(market_stats['median'])} median total cost fallback)"
    )
    print(f"\n\U0001f50e Searching active listings with total cost under {format_currency(max_total_cost)} {threshold_message}...\n")

    while len(all_items) < max_items:
        params = {
            "q": full_query,
            "filter": (
                f"buyingOptions:{{FIXED_PRICE}},conditions:{{{condition_filter}}},"
                f"price:[..{max_total_cost}],priceCurrency:USD"
            ),
            "sort": "price",
            "limit": str(RESULTS_PER_PAGE),
            "offset": str(page * RESULTS_PER_PAGE),
        }

        try:
            result = browse_search(params, f"Browse deal search page {page + 1}")
        except RETRYABLE_EXCEPTIONS as exc:
            if not all_items:
                raise RuntimeError(f"Unable to fetch deal search page {page + 1}: {exc}") from exc
            print(
                f"\u26a0 Deal search stopped at page {page + 1}: {exc}. "
                f"Continuing with {len(all_items)} result(s)."
            )
            break

        items = result.get("itemSummaries") or []
        if not items:
            break

        all_items.extend(items)
        total = result.get("total", 0)
        if len(all_items) >= total:
            break

        page += 1

    if not all_items:
        print("\U0001f615 No deals found right now. Try again later!\n")
        return []

    deals = []
    filtered_out = 0
    for item in all_items:
        cost_info = extract_total_cost(item)
        price = cost_info["price"]
        shipping_cost = cost_info["shipping_cost"]
        total_cost = cost_info["total_cost"]
        shipping_type = cost_info["shipping_type"]

        if price <= 0:
            continue

        # Post-filter: the API price filter only checks item price, not total cost
        if total_cost > max_total_cost:
            filtered_out += 1
            continue

        discount_pct = ((market_stats["median"] - total_cost) / market_stats["median"]) * 100
        z_score = ((total_cost - market_stats["median"]) / market_stats["std_dev"]) if has_usable_std_dev else None
        price_score = z_score if has_usable_std_dev else -discount_pct
        gross_profit = market_stats["median"] - total_cost
        fees = market_stats["median"] * EBAY_FEE_RATE
        net_profit = gross_profit - fees

        if has_usable_std_dev:
            if z_score <= -2:
                signal = "\U0001f525 Strong buy"
            elif z_score <= -1:
                signal = "\u2705 Buy"
            else:
                signal = "\u26a0\ufe0f  Marginal"
        else:
            if discount_pct >= 20:
                signal = "\U0001f525 Strong buy"
            elif discount_pct >= 10:
                signal = "\u2705 Buy"
            else:
                signal = "\u26a0\ufe0f  Marginal"

        deals.append(
            {
                "item_id": item.get("itemId", ""),
                "title": item.get("title", ""),
                "price": price,
                "shipping_cost": shipping_cost,
                "shipping_type": shipping_type,
                "total_cost": total_cost,
                "z_score": z_score,
                "discount_pct": discount_pct,
                "price_score": price_score,
                "signal": signal,
                "gross_profit": gross_profit,
                "net_profit": net_profit,
                "link": item.get("itemWebUrl", ""),
            }
        )

    deals.sort(key=lambda d: d["price_score"])
    if filtered_out:
        print(f"\U0001f4e6 Found {len(deals)} deal(s) across {page + 1} page(s) ({filtered_out} excluded by total cost post-filter)")
    else:
        print(f"\U0001f4e6 Found {len(deals)} deal(s) across {page + 1} page(s)")
    return deals

def enrich_deals(deals, market_stats):
    has_usable_std_dev = market_stats["has_usable_std_dev"]
    print(f"\n\U0001f4e6 Enriching {len(deals)} deals with sold quantity and shipping data...\n")
    for deal in deals:
        item_id = deal.get("item_id")
        if not item_id:
            deal["sold_quantity"] = None
            continue

        try:
            details = get_item_details(item_id)
            availabilities = details.get("estimatedAvailabilities") or []
            sold_qty = availabilities[0].get("estimatedSoldQuantity", 0) if availabilities else 0
            deal["sold_quantity"] = sold_qty if isinstance(sold_qty, int) else 0

            # Refine shipping cost from getItem if search data was incomplete
            if deal.get("shipping_type") in ("UNKNOWN", "FREE_OR_UNKNOWN"):
                detail_cost = extract_total_cost(details)
                if detail_cost["shipping_cost"] > 0 or detail_cost["shipping_type"] not in ("UNKNOWN",):
                    deal["shipping_cost"] = detail_cost["shipping_cost"]
                    deal["shipping_type"] = detail_cost["shipping_type"]
                    deal["total_cost"] = deal["price"] + deal["shipping_cost"]
                    # Recalculate metrics with refined shipping
                    deal["discount_pct"] = ((market_stats["median"] - deal["total_cost"]) / market_stats["median"]) * 100
                    deal["z_score"] = ((deal["total_cost"] - market_stats["median"]) / market_stats["std_dev"]) if has_usable_std_dev else None
                    deal["price_score"] = deal["z_score"] if has_usable_std_dev else -deal["discount_pct"]
                    deal["gross_profit"] = market_stats["median"] - deal["total_cost"]
                    fees = market_stats["median"] * EBAY_FEE_RATE
                    deal["net_profit"] = deal["gross_profit"] - fees
        except RETRYABLE_EXCEPTIONS as exc:
            deal["sold_quantity"] = None
            print(f"\u26a0 Enrichment failed for {item_id}: {exc}")

    for deal in deals:
        sold = deal["sold_quantity"] if isinstance(deal.get("sold_quantity"), int) else 0
        deal["composite_score"] = deal["price_score"] - 0.1 * sold

    deals.sort(key=lambda d: d["composite_score"])
    print("\u2705 Enrichment complete")
    return deals

print("\u2705 Deal discovery and enrichment functions loaded")


✅ Deal discovery and enrichment functions loaded


## 6) Optional Azure OpenAI Listing Evaluation


In [15]:
VALID_VERDICTS = {"BUY", "RISKY", "PASS"}

LLM_STATE_DISABLED = "DISABLED"
LLM_STATE_SKIPPED_BUDGET = "SKIPPED_BUDGET"
LLM_STATE_EVALUATED = "EVALUATED"
LLM_STATE_ERROR = "ERROR"
LLM_STATE_UNSET = "UNSET"

def to_list(value):
    if value is None:
        return []
    return value if isinstance(value, list) else [value]

def parse_llm_response(content):
    if not content or not content.strip():
        raise ValueError("LLM returned an empty response.")

    parsed = json.loads(content)
    verdict = str(parsed.get("verdict", "")).upper()
    confidence = float(parsed.get("confidence", -1))
    issues = parsed.get("issues", [])
    summary = str(parsed.get("summary", "")).strip()

    if verdict not in VALID_VERDICTS:
        raise ValueError(f"Invalid verdict: {parsed.get('verdict')}")
    if not (0 <= confidence <= 100):
        raise ValueError(f"Invalid confidence: {parsed.get('confidence')}")
    if not isinstance(issues, list) or any(not isinstance(issue, str) for issue in issues):
        raise ValueError("Invalid issues payload.")
    if not summary:
        raise ValueError("Invalid summary payload.")

    return {
        "verdict": verdict,
        "confidence": round(confidence),
        "issues": [issue.strip() for issue in issues if issue.strip()],
        "summary": summary,
    }

def get_listing_details(item_id):
    item = get_item_details(item_id)
    image_urls = [item.get("image", {}).get("imageUrl")]
    image_urls.extend(img.get("imageUrl") for img in to_list(item.get("additionalImages")) if isinstance(img, dict))

    return {
        "title": item.get("title", ""),
        "description": item.get("description", "") or item.get("shortDescription", ""),
        "image_urls": [url for url in image_urls if url],
        "condition_description": item.get("conditionDescription", ""),
        "condition_name": item.get("condition", ""),
        "item_specifics": [
            {
                "name": aspect.get("name", ""),
                "value": to_list(aspect.get("value")),
            }
            for aspect in to_list(item.get("localizedAspects"))
            if isinstance(aspect, dict)
        ],
        "seller": {
            "user_id": item.get("seller", {}).get("username", ""),
            "feedback_score": item.get("seller", {}).get("feedbackScore", 0),
            "positive_feedback_pct": item.get("seller", {}).get("feedbackPercentage", 0),
        },
    }

def evaluate_listing_with_llm(client, listing_details, deal_context):
    plain_description = re.sub(r"<[^>]*>", " ", listing_details["description"])
    plain_description = re.sub(r"\s+", " ", plain_description).strip()[:3000]

    item_specifics_text = "\n".join(
        f"{spec['name']}: {', '.join(spec['value']) if isinstance(spec['value'], list) else spec['value']}"
        for spec in listing_details["item_specifics"]
    )

    z_context = (
        f"{deal_context['z_score']:.2f} (negative = below market)"
        if deal_context["z_score"] is not None
        else "N/A (market variance too low for z-score)"
    )

    content = [
        {
            "type": "text",
            "text": (
                f"You are an expert eBay arbitrage analyst evaluating listings for '{SEARCH_QUERY}' resale potential.\n\n"
                f"VERDICT RULES — follow these strictly:\n\n"
                f"PASS (automatic reject) — use ONLY for these two reasons:\n"
                f"  1. WRONG PRODUCT: The listing is NOT a '{SEARCH_QUERY}'. It is a different model, brand, "
                f"an accessory/case/charger only, or a bundle where the primary item is not '{SEARCH_QUERY}'.\n"
                f"  2. NON-FUNCTIONAL: The listing is explicitly sold as 'for parts', 'for parts or not working', "
                f"'as-is non-functional', or the seller states the item does not power on or is broken beyond use.\n\n"
                f"RISKY (proceed with caution) — use when the item IS a '{SEARCH_QUERY}' and IS functional/working, but has concerns:\n"
                f"  - Cosmetic damage (scratches, screen marks, worn keys, discoloration)\n"
                f"  - Missing accessories (no charger, no cover, no cable)\n"
                f"  - Vague or sparse description, few photos\n"
                f"  - Low seller feedback or credibility concerns\n"
                f"  - Condition described as 'acceptable' or 'fair' with visible wear\n\n"
                f"BUY (recommended) — use when the item IS a '{SEARCH_QUERY}', IS functional, and looks good for resale:\n"
                f"  - Product identity confirmed, photos match\n"
                f"  - Working condition, reasonable cosmetic state\n"
                f"  - No major red flags from seller or description\n\n"
                f"IMPORTANT: If the listing IS a '{SEARCH_QUERY}' and the item IS described as working/functional, "
                f"do NOT return PASS. Use RISKY for condition concerns, BUY if it looks solid.\n"
                f"Only use PASS for wrong-product or non-functional/for-parts listings.\n\n"
                f"LISTING DETAILS:\n"
                f"- Title: {listing_details['title']}\n"
                f"- Condition: {listing_details['condition_name']}\n"
                f"- Condition Notes: {listing_details['condition_description'] or 'None provided'}\n"
                f"- Seller: {listing_details['seller']['user_id']} ({listing_details['seller']['positive_feedback_pct']}% positive, {listing_details['seller']['feedback_score']} feedback score)\n"
                f"- Item Specifics:\n{item_specifics_text or 'None listed'}\n\n"
                f"DESCRIPTION:\n{plain_description or 'No description provided'}\n\n"
                f"DEAL CONTEXT:\n"
                f"- Listed price: ${deal_context['price']:.2f}\n"
                f"- Shipping cost: ${deal_context['shipping_cost']:.2f} ({deal_context['shipping_type']})\n"
                f"- Total cost (price + shipping): ${deal_context['total_cost']:.2f}\n"
                f"- Market median (total cost): ${deal_context['market_median']:.2f}\n"
                f"- Z-score: {z_context}\n"
                f"- Discount vs median: {deal_context['discount_pct']:.1f}% below median\n"
                f"- Estimated net profit (incl. shipping): ${deal_context['net_profit']:.2f}\n\n"
                f"Evaluate product identity first, then image condition, description red flags, seller credibility, accessory completeness, and resale suitability."
            ),
        }
    ]

    for image_url in listing_details["image_urls"][:MAX_IMAGES_PER_LISTING]:
        content.append({"type": "image_url", "image_url": {"url": image_url}})

    def _request(request_timeout_s):
        return client.chat.completions.create(
            model=AZURE_OPENAI_DEPLOYMENT,
            messages=[{"role": "user", "content": content}],
            max_completion_tokens=800,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "listing_evaluation",
                    "strict": True,
                    "schema": {
                        "type": "object",
                        "properties": {
                            "verdict": {"type": "string", "enum": ["BUY", "RISKY", "PASS"]},
                            "confidence": {"type": "number", "minimum": 0, "maximum": 100},
                            "issues": {"type": "array", "items": {"type": "string"}},
                            "summary": {"type": "string"},
                        },
                        "required": ["verdict", "confidence", "issues", "summary"],
                        "additionalProperties": False,
                    },
                },
            },
            timeout=request_timeout_s,
        )

    response = call_with_retry(
        f"Azure OpenAI listing evaluation for {listing_details['title'][:30] or 'listing'}",
        _request,
        LLM_TIMEOUT_S,
    )

    return parse_llm_response(response.choices[0].message.content)

def evaluate_deals_with_llm(deals, market_stats):
    for deal in deals:
        deal["llm_state"] = LLM_STATE_UNSET
        deal["llm_verdict"] = "N/A"
        deal["llm_confidence"] = 0
        deal["llm_issues"] = []
        deal["llm_summary"] = "Not evaluated"

    if not LLM_ENABLED:
        for deal in deals:
            deal["llm_state"] = LLM_STATE_DISABLED
            deal["llm_summary"] = "LLM disabled by configuration"
        print(
            "\n💡 LLM evaluation skipped — set AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, "
            "and AZURE_OPENAI_DEPLOYMENT in .env to enable.\n"
        )
        return deals

    client = AzureOpenAI(
        api_version="2024-10-01-preview",
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
    )

    to_evaluate = deals[:MAX_LLM_CALLS]
    skipped = deals[MAX_LLM_CALLS:]
    for deal in skipped:
        deal["llm_state"] = LLM_STATE_SKIPPED_BUDGET
        deal["llm_summary"] = "Skipped due LLM budget cap"

    if skipped:
        print(
            f"  ℹ️  {len(deals)} deals passed the statistical filter; evaluating top {len(to_evaluate)} "
            f"(budget cap: {MAX_LLM_CALLS})."
        )
        print(
            f"     Remaining {len(skipped)} deal(s) will use price-signal-only recommendations.\n"
        )

    print(f"\n🤖 Evaluating {len(to_evaluate)} deals with Azure OpenAI ({LLM_CONCURRENCY} parallel)...\n")

    progress_lock = Lock()
    completed = [0]

    def evaluate_one(deal, index):
        label = f"  [{index + 1}/{len(to_evaluate)}] {deal['title'][:40]}..."
        try:
            listing_details = get_listing_details(deal["item_id"])
            evaluation = evaluate_listing_with_llm(
                client,
                listing_details,
                {
                    "price": deal["price"],
                    "shipping_cost": deal.get("shipping_cost", 0),
                    "shipping_type": deal.get("shipping_type", "UNKNOWN"),
                    "total_cost": deal.get("total_cost", deal["price"]),
                    "market_median": market_stats["median"],
                    "z_score": deal["z_score"],
                    "discount_pct": deal["discount_pct"],
                    "net_profit": deal["net_profit"],
                },
            )
            deal["llm_state"] = LLM_STATE_EVALUATED
            deal["llm_verdict"] = evaluation["verdict"]
            deal["llm_confidence"] = evaluation["confidence"]
            deal["llm_issues"] = evaluation["issues"]
            deal["llm_summary"] = evaluation["summary"]

            icon = {"BUY": "🟢", "RISKY": "🟡", "PASS": "🔴"}[evaluation["verdict"]]
            with progress_lock:
                completed[0] += 1
                progress = completed[0]
            print(f"{label} {icon} {evaluation['verdict']} ({evaluation['confidence']}%) [{progress}/{len(to_evaluate)}]")
        except (
            ValueError,
            KeyError,
            TypeError,
            requests.exceptions.RequestException,
            APITimeoutError,
            APIConnectionError,
            RateLimitError,
            APIStatusError,
        ) as exc:
            deal["llm_state"] = LLM_STATE_ERROR
            deal["llm_verdict"] = "N/A"
            deal["llm_confidence"] = 0
            deal["llm_issues"] = []
            deal["llm_summary"] = f"Error: {exc}"
            with progress_lock:
                completed[0] += 1
                progress = completed[0]
            print(f"{label} ⚪ Error: {str(exc)[:60]} [{progress}/{len(to_evaluate)}]")

    with ThreadPoolExecutor(max_workers=LLM_CONCURRENCY) as executor:
        futures = [executor.submit(evaluate_one, deal, idx) for idx, deal in enumerate(to_evaluate)]
        for future in as_completed(futures):
            future.result()

    return deals

print("✅ LLM evaluation functions loaded")


✅ LLM evaluation functions loaded


## 7) Final Ranking, Recommendations, and Display


In [16]:
def apply_recommendations(deals):
    for deal in deals:
        price_rank = price_rank_from_signal(deal.get("signal", ""))
        llm_state = deal.get("llm_state", LLM_STATE_UNSET)
        llm_verdict = deal.get("llm_verdict", "N/A")
        confidence = deal.get("llm_confidence", 0)

        llm_rank = -1
        if llm_state == LLM_STATE_EVALUATED:
            llm_rank = {"BUY": 3, "RISKY": 2, "PASS": 0}.get(llm_verdict, -1)

        llm_bonus = llm_rank if llm_rank > 0 else 0
        confidence_bonus = (confidence / 100) if llm_state == LLM_STATE_EVALUATED else 0
        deal["final_score"] = price_rank + llm_bonus + confidence_bonus

        if llm_state in (LLM_STATE_DISABLED, LLM_STATE_SKIPPED_BUDGET, LLM_STATE_UNSET):
            deal["final_rec"] = price_only_recommendation(price_rank)
        elif llm_state == LLM_STATE_ERROR:
            deal["final_rec"] = "\u26a0\ufe0f  CONSIDER" if price_rank >= 2 else "\u274c SKIP"
        elif llm_rank == 0:
            deal["final_rec"] = "\u274c SKIP"
        elif llm_rank == 3 and price_rank >= 2:
            deal["final_rec"] = "\U0001f3c6 STRONG BUY"
        elif llm_rank == 3:
            deal["final_rec"] = "\u2705 BUY"
        elif llm_rank == 2 and price_rank >= 2:
            deal["final_rec"] = "\u26a0\ufe0f  CONSIDER"
        else:
            deal["final_rec"] = "\u274c SKIP"

    REC_ORDER = {"\U0001f3c6 STRONG BUY": 0, "\u2705 BUY": 1, "\u26a0\ufe0f  CONSIDER": 2, "\u274c SKIP": 3}
    deals.sort(key=lambda d: (REC_ORDER.get(d["final_rec"], 9), -d["final_score"]))
    return deals

def format_shipping(deal):
    sc = deal.get("shipping_cost", 0)
    st = deal.get("shipping_type", "UNKNOWN")
    if st in ("UNKNOWN",):
        return "?"
    if sc == 0:
        return "FREE"
    return format_currency(sc)

def _rec_color(rec):
    if "STRONG BUY" in rec:
        return "background-color: #c6efce; color: #006100"
    if "BUY" in rec:
        return "background-color: #dff0d8; color: #3c763d"
    if "CONSIDER" in rec:
        return "background-color: #fcf8e3; color: #8a6d3b"
    return "background-color: #f2dede; color: #a94442"

def display_results(deals, market_stats):
    deals = apply_recommendations(deals)
    has_usable = market_stats["has_usable_std_dev"]
    show_llm = any(d.get("llm_state") != LLM_STATE_DISABLED for d in deals)

    rows = []
    for deal in deals:
        score_val = (
            f"{deal['z_score']:.2f}" if has_usable and deal.get("z_score") is not None
            else f"{deal['discount_pct']:.1f}%"
        )
        sold = deal["sold_quantity"] if isinstance(deal.get("sold_quantity"), int) else "?"
        ship = format_shipping(deal)
        total = deal.get("total_cost", deal.get("price", 0))
        profit = deal["net_profit"]
        profit_str = f"+{format_currency(profit)}" if profit > 0 else format_currency(profit)

        # LLM columns
        verdict_str = ""
        llm_summary_str = ""
        if show_llm:
            state = deal.get("llm_state", LLM_STATE_UNSET)
            if state == LLM_STATE_EVALUATED:
                icon = {"BUY": "\U0001f7e2", "RISKY": "\U0001f7e1", "PASS": "\U0001f534"}.get(deal.get("llm_verdict"), "\u26aa")
                verdict_str = f"{icon} {deal.get('llm_verdict', 'N/A')}"
            elif state == LLM_STATE_SKIPPED_BUDGET:
                verdict_str = "\u26aa BUDGET"
            elif state == LLM_STATE_ERROR:
                verdict_str = "\u26aa ERROR"
            else:
                verdict_str = "\u26aa N/A"

            summary = deal.get("llm_summary", "")
            if summary and not summary.startswith("Not evaluated") and not summary.startswith("LLM disabled"):
                llm_summary_str = summary[:200]
            elif summary:
                llm_summary_str = summary

        title = deal["title"][:60]
        link = deal.get("link", "")

        row = {
            "Rec": deal["final_rec"],
            "Score": score_val,
            "Sold": sold,
        }
        if show_llm:
            row["Verdict"] = verdict_str
        row.update({
            "Signal": deal["signal"],
            "Price": format_currency(deal["price"]),
            "Ship": ship,
            "Total": format_currency(total),
            "Net Profit": profit_str,
            "Title": title,
        })
        if show_llm:
            row["LLM Summary"] = llm_summary_str
        row["Link"] = link
        rows.append(row)

    df = pd.DataFrame(rows)

    def style_row(row):
        color = _rec_color(row["Rec"])
        return [color] * len(row)

    def make_clickable(url):
        if not url:
            return ""
        return f'<a href="{url}" target="_blank">View</a>'

    styled = (
        df.style
        .apply(style_row, axis=1)
        .format({"Link": make_clickable}, escape="html")
        .set_properties(**{
            "text-align": "left",
            "white-space": "normal",
            "word-wrap": "break-word",
        })
        .set_properties(subset=["Title"], **{"min-width": "200px", "max-width": "350px"})
        .set_table_styles([
            {"selector": "th", "props": [("background-color", "#f5f5f5"), ("font-weight", "bold"), ("text-align", "left"), ("padding", "6px 8px")]},
            {"selector": "td", "props": [("padding", "6px 8px"), ("vertical-align", "top")]},
        ])
        .hide(axis="index")
    )

    if show_llm and "LLM Summary" in df.columns:
        styled = styled.set_properties(subset=["LLM Summary"], **{"min-width": "250px", "max-width": "400px", "font-size": "12px"})

    print(f"\n\U0001f4ca Full Analysis \u2014 {len(deals)} deals (best first):\n")
    display(styled)

    # Column guide
    score_label = "Score = (total cost - median) / std dev" if has_usable else "Score = ((median - total cost) / median) \u00d7 100"
    print(f"\n\U0001f4a1 {score_label}")
    print(f"   NET PROFIT = (median - total cost) minus ~{EBAY_FEE_RATE * 100:.0f}% eBay fees")

    # Action items summary
    actionable = [d for d in deals if d["final_rec"] in ("\U0001f3c6 STRONG BUY", "\u2705 BUY")]
    consider = [d for d in deals if d["final_rec"] == "\u26a0\ufe0f  CONSIDER"]
    skipped = sum(1 for d in deals if d["final_rec"] == "\u274c SKIP")

    print(f"\n\u2550" * 70)
    print("  \U0001f3af ACTION ITEMS \u2014 Listings to buy NOW")
    print("\u2550" * 70)
    if not actionable:
        print("\n  No strong recommendations right now.")
    else:
        for i, deal in enumerate(actionable, start=1):
            score_summary = (
                f"z={deal['z_score']:.2f}" if has_usable and deal.get("z_score") is not None else f"{deal['discount_pct']:.1f}% off"
            )
            ship_label = format_shipping(deal)
            print(f"\n  {deal['final_rec']}  [{i}]")
            print(f"  \U0001f4e6 {deal['title'][:70]}")
            print(
                f"  \U0001f4b0 {format_currency(deal['price'])} + {ship_label} shipping = {format_currency(deal.get('total_cost', deal['price']))} total"
            )
            print(
                f"     \u2192 est. profit "
                f"{('+' if deal['net_profit'] > 0 else '')}{format_currency(deal['net_profit'])} ({score_summary})"
            )
            if deal.get("llm_summary") and not deal["llm_summary"].startswith("Not evaluated"):
                print(f"  \U0001f916 {deal['llm_summary']}")
            print(f"  \U0001f517 {deal.get('link', '')}")

    if consider:
        print("\n" + "\u2500" * 70)
        print("  \u26a0\ufe0f  WORTH CONSIDERING (proceed with caution)")
        print("\u2500" * 70)
        for i, deal in enumerate(consider, start=1):
            score_summary = (
                f"z={deal['z_score']:.2f}" if has_usable and deal.get("z_score") is not None else f"{deal['discount_pct']:.1f}% off"
            )
            ship_label = format_shipping(deal)
            print(f"\n  [{i}] {deal['title'][:65]}")
            print(
                f"      {format_currency(deal['price'])} + {ship_label} ship = {format_currency(deal.get('total_cost', deal['price']))} | profit: "
                f"{('+' if deal['net_profit'] > 0 else '')}{format_currency(deal['net_profit'])} ({score_summary})"
            )
            if deal.get("llm_summary") and not deal["llm_summary"].startswith("Not evaluated"):
                print(f"      \U0001f916 {deal['llm_summary']}")
            for issue in deal.get("llm_issues", []):
                print(f"      \u26a0 {issue}")
            print(f"      \U0001f517 {deal.get('link', '')}")

    print(f"\n  \U0001f4ca Summary: {len(actionable)} BUY | {len(consider)} CONSIDER | {skipped} SKIP out of {len(deals)} deals")
    print("\u2550" * 70 + "\n")

print("\u2705 Recommendation and display functions loaded")


✅ Recommendation and display functions loaded


## 8) Execute Full Pipeline


In [17]:
print("═" * 50)
print("  eBay Deal Finder — Python Notebook")
print("═" * 50)
print(f"  Query:     '{SEARCH_QUERY}'")
print(f"  Condition: {CONDITION}")
print(f"  Min z:     {MIN_Z_SCORE} ({abs(MIN_Z_SCORE)}σ below median)")
print("═" * 50)

clear_item_details_cache()

market_items = get_market_listings(SEARCH_QUERY, CONDITION)
if not market_items:
    raise RuntimeError("No market listings found. Check your query or API credentials.")

market_stats = analyze_prices(market_items)
if not market_stats:
    raise RuntimeError("Market analysis did not produce usable pricing statistics.")

deals = find_deal_listings(SEARCH_QUERY, CONDITION, market_stats)
if deals:
    deals = enrich_deals(deals, market_stats)
    deals = evaluate_deals_with_llm(deals, market_stats)
    display_results(deals, market_stats)
else:
    print("No qualifying deals returned for the configured threshold.")


══════════════════════════════════════════════════
  eBay Deal Finder — Python Notebook
══════════════════════════════════════════════════
  Query:     '"TI-84 Plus CE" graphing calculator'
  Condition: Used
  Min z:     -1.0 (1.0σ below median)
══════════════════════════════════════════════════

📊 Fetching active market listings for '"TI-84 Plus CE" graphing calculator' (Used)...

──────────────────────────────────────────────────
  RAW MARKET TOTAL COSTS (price + shipping)
──────────────────────────────────────────────────
  Items analyzed:    420
  Average total:     $90.49
  Median total:      $82.08
  Low / High:        $10.50 — $899.00
  Avg shipping:      $6.71
  Median shipping:   $7.46
──────────────────────────────────────────────────

  🧹 IQR FILTER: keeping $32.06 — $139.79
     Removed 22 outlier(s)

──────────────────────────────────────────────────
  FILTERED MARKET VALUE (total cost = price + shipping)
──────────────────────────────────────────────────
  Items kept:    

Rec,Score,Sold,Verdict,Signal,Price,Ship,Total,Net Profit,Title,LLM Summary,Link
🏆 STRONG BUY,-1.34,0,🟢 BUY,✅ Buy,$49.99,$5.62,$55.61,+$15.30,Texas Instruments TI-84 Plus CE Graphing Calculator Blue,"Listing matches the TI-84 Plus CE graphing calculator, and the photos confirm the correct model and a working screen/display. The item is functional and not sold for parts. Missing the cover is a mino",View
🏆 STRONG BUY,-1.06,0,🟢 BUY,✅ Buy,$55.00,$6.00,$61.00,+$9.91,Texas Instruments TI-84 Plus CE Graphing Calculator Black Wh,"This is the correct TI-84 Plus CE graphing calculator, and the photos show it powers on and tests working. It includes the charger, so accessory completeness is good. Cosmetic wear is present, especia",View
🏆 STRONG BUY,-1.06,0,🟢 BUY,✅ Buy,$54.99,$6.01,$61.00,+$9.91,Texas Instruments TI-84 Plus CE Graphing Calculator Programm,"The listing is clearly a TI-84 Plus CE graphing calculator, and the photos match the model. The seller states it is fully tested, working, sanitized, and charged, with the case included. Cosmetic wear",View
⚠️ CONSIDER,-3.67,3,🟡 RISKY,🔥 Strong buy,$10.50,FREE,$10.50,+$60.41,Texas Instrument TI 84 Plus CE Ti Nspire CX II CAS Graphing,"This is a functional TI-84 Plus CE-compatible data cable listing, not a calculator bundle. The item appears to be the correct accessory and is tested working, but the title is confusing and the photos",View
⚠️ CONSIDER,-3.18,0,🟡 RISKY,🔥 Strong buy,$20.00,FREE,$20.00,+$50.91,Texas Instruments TI 84 Plus CE Ti Nspire CX II CAS Graphing,"Photos confirm a TI-84 Plus CE graphing calculator with battery and cover, and the seller claims it is working. However, the title/description are inconsistent and somewhat misleading, and the cosmeti",View
⚠️ CONSIDER,-2.64,0,🟡 RISKY,🔥 Strong buy,$23.00,$7.46,$30.46,+$40.45,Texas Instruments TI-84 Plus CE Color Graphing Calculator -,"This appears to be a TI-84 Plus CE graphing calculator, and it is not listed as for parts or non-working, so it does not qualify for PASS. However, the screen glitch plus missing covers and visible we",View
⚠️ CONSIDER,-2.33,0,🟡 RISKY,🔥 Strong buy,$28.00,$8.46,$36.46,+$34.45,Texas Instruments TI-84 Plus CE Color Graphing Calculator -,"This appears to be the correct TI-84 Plus CE graphing calculator and it powers on, so it is not a pass. The listing has solid resale potential at the shown price, but cosmetic blemishes, missing case,",View
⚠️ CONSIDER,-2.02,0,🟡 RISKY,🔥 Strong buy,$35.00,$7.46,$42.46,+$28.45,Texas Instruments TI-84 Plus CE Color Graphing Calculator -,"The listing appears to be a working Texas Instruments TI-84 Plus calculator, but it is not clearly the TI-84 Plus CE model requested. Because the item is functional but has a broken battery cover clip",View
⚠️ CONSIDER,-1.89,0,🟡 RISKY,✅ Buy,$35.66,$9.26,$44.92,+$25.99,Texas Instruments TI-84 Plus CE Color Graphing Calculator -,"The listing appears to be a Texas Instruments TI-84 graphing calculator in working condition, and the photo shows the calculator powered on. However, the identity is inconsistent because the photo sho",View
⚠️ CONSIDER,-1.88,0,🟡 RISKY,✅ Buy,$45.00,FREE,$45.00,+$25.91,Texas Instruments TI-84 Plus CE Graphing Calculator Color Pi,"This appears to be a functional Texas Instruments TI-84 calculator and is not a for-parts or wrong-product listing, but the model mismatch between the title (TI-84 Plus CE) and the photo showing TI-84",View



💡 Score = (total cost - median) / std dev
   NET PROFIT = (median - total cost) minus ~13% eBay fees

═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
  🎯 ACTION ITEMS — Listings to buy NOW
══════════════════════════════════════════════════════════════════════

  🏆 STRONG BUY  [1]
  📦 Texas Instruments TI-84 Plus CE Graphing Calculator  Blue
  💰 $49.99 + $5.62 shipping = $55.61 total
     → est. profit +$15.30 (z=-1.34)
  🤖 Listing matches the TI-84 Plus CE graphing calculator, and the photos confirm the correct model and a working screen/display. The item is functional and not sold for parts. Missing the cover is a minor accessory issue, and the unit shows some cosmetic wear, but overall the below-market total cost and positive estimated profit make it a solid resale candidate.
  🔗 https://www.ebay.com/itm/366318151808?_skw=%22TI-84+Plus+CE%22+graphing+calculator+-charger+-case+-cable+-cover+-e

## 9) Deterministic Sanity Checks (No Network)


In [18]:
def run_deterministic_checks():
    outliers = filter_outliers([10, 11, 12, 13, 14, 999])
    assert 999 not in outliers["filtered"], "Outlier filter should remove obvious outliers"

    # --- Test extract_total_cost ---
    # Free shipping
    free_ship_item = {"price": {"value": "50.00"}, "shippingOptions": [{"shippingCost": {"value": "0.00", "currency": "USD"}, "shippingCostType": "FIXED"}]}
    result = extract_total_cost(free_ship_item)
    assert result["price"] == 50.0
    assert result["shipping_cost"] == 0.0
    assert result["total_cost"] == 50.0
    assert result["shipping_type"] == "FREE"

    # Paid shipping
    paid_ship_item = {"price": {"value": "40.00"}, "shippingOptions": [{"shippingCost": {"value": "8.50", "currency": "USD"}, "shippingCostType": "FIXED"}]}
    result = extract_total_cost(paid_ship_item)
    assert result["price"] == 40.0
    assert result["shipping_cost"] == 8.50
    assert result["total_cost"] == 48.50
    assert result["shipping_type"] == "FIXED"

    # Missing shipping options
    no_ship_item = {"price": {"value": "30.00"}}
    result = extract_total_cost(no_ship_item)
    assert result["price"] == 30.0
    assert result["shipping_cost"] == 0.0
    assert result["total_cost"] == 30.0
    assert result["shipping_type"] == "UNKNOWN"

    # CALCULATED shipping with a value
    calc_ship_item = {"price": {"value": "60.00"}, "shippingOptions": [{"shippingCost": {"value": "12.00", "currency": "USD"}, "shippingCostType": "CALCULATED"}]}
    result = extract_total_cost(calc_ship_item)
    assert result["price"] == 60.0
    assert result["shipping_cost"] == 12.0
    assert result["total_cost"] == 72.0
    assert result["shipping_type"] == "CALCULATED"

    print("\u2705 extract_total_cost tests passed")

    # --- Test net profit includes shipping ---
    # median=100, price=40, shipping=8.50, total_cost=48.50, fee_rate=0.13
    median = 100.0
    total_cost = 48.50
    gross_profit = median - total_cost  # 51.50
    fees = median * EBAY_FEE_RATE  # 13.00
    net_profit = gross_profit - fees  # 38.50
    assert abs(net_profit - 38.50) < 0.01, f"Net profit should be 38.50 but got {net_profit}"
    print("\u2705 Shipping-aware profit calculation verified")

    # --- Test apply_recommendations with shipping fields ---
    sample_deals = [
        {
            "id": "disabled-strong",
            "signal": "\U0001f525 Strong buy",
            "llm_state": LLM_STATE_DISABLED,
            "llm_verdict": "N/A",
            "llm_confidence": 0,
            "discount_pct": 25,
        },
        {
            "id": "budget-buy",
            "signal": "\u2705 Buy",
            "llm_state": LLM_STATE_SKIPPED_BUDGET,
            "llm_verdict": "N/A",
            "llm_confidence": 0,
            "discount_pct": 12,
        },
        {
            "id": "pass-strong",
            "signal": "\U0001f525 Strong buy",
            "llm_state": LLM_STATE_EVALUATED,
            "llm_verdict": "PASS",
            "llm_confidence": 99,
            "discount_pct": 30,
        },
        {
            "id": "error-buy",
            "signal": "\u2705 Buy",
            "llm_state": LLM_STATE_ERROR,
            "llm_verdict": "N/A",
            "llm_confidence": 0,
            "discount_pct": 11,
        },
        {
            "id": "eval-buy",
            "signal": "\u2705 Buy",
            "llm_state": LLM_STATE_EVALUATED,
            "llm_verdict": "BUY",
            "llm_confidence": 90,
            "discount_pct": 15,
        },
    ]

    for sample in sample_deals:
        sample["price"] = 50
        sample["shipping_cost"] = 5.0
        sample["shipping_type"] = "FIXED"
        sample["total_cost"] = 55.0
        sample["z_score"] = -1.2
        sample["sold_quantity"] = 0
        sample["title"] = sample["id"]
        sample["net_profit"] = 10
        sample["link"] = ""
        sample["llm_issues"] = []
        sample["llm_summary"] = ""

    ranked = apply_recommendations(sample_deals)
    by_id = {deal["id"]: deal for deal in ranked}

    assert by_id["disabled-strong"]["final_rec"] == "\u2705 BUY"
    assert by_id["budget-buy"]["final_rec"] == "\u26a0\ufe0f  CONSIDER"
    assert by_id["pass-strong"]["final_rec"] == "\u274c SKIP"
    assert by_id["error-buy"]["final_rec"] == "\u26a0\ufe0f  CONSIDER"
    assert by_id["eval-buy"]["final_rec"] in ("\U0001f3c6 STRONG BUY", "\u2705 BUY")

    # --- Test product mismatch override: PASS + great z-score = SKIP ---
    mismatch_deal = {
        "id": "mismatch-great-price",
        "signal": "\U0001f525 Strong buy",
        "llm_state": LLM_STATE_EVALUATED,
        "llm_verdict": "PASS",
        "llm_confidence": 95,
        "discount_pct": 40,
        "price": 30,
        "shipping_cost": 0,
        "shipping_type": "FREE",
        "total_cost": 30,
        "z_score": -3.0,
        "sold_quantity": 10,
        "title": "Wrong product - great price",
        "net_profit": 40,
        "link": "",
        "llm_issues": ["Product mismatch: listing is TI-Nspire, not TI-84 Plus CE"],
        "llm_summary": "Wrong product entirely",
    }
    mismatch_result = apply_recommendations([mismatch_deal])
    assert mismatch_result[0]["final_rec"] == "\u274c SKIP", (
        f"Product mismatch (PASS verdict) should always be SKIP, got {mismatch_result[0]['final_rec']}"
    )
    print("\u2705 Product mismatch override verified (PASS + z=-3.0 = SKIP)")

    # --- Test RISKY verdict: correct product with issues should NOT be killed ---
    risky_deal = {
        "id": "risky-good-price",
        "signal": "\U0001f525 Strong buy",
        "llm_state": LLM_STATE_EVALUATED,
        "llm_verdict": "RISKY",
        "llm_confidence": 60,
        "discount_pct": 30,
        "price": 40,
        "shipping_cost": 5.0,
        "shipping_type": "FIXED",
        "total_cost": 45.0,
        "z_score": -2.0,
        "sold_quantity": 0,
        "title": "Correct product with scratches",
        "net_profit": 25,
        "link": "",
        "llm_issues": ["Cosmetic scratches on screen"],
        "llm_summary": "Correct TI-84 Plus CE but has screen scratches",
    }
    risky_result = apply_recommendations([risky_deal])
    assert risky_result[0]["final_rec"] == "\u26a0\ufe0f  CONSIDER", (
        f"RISKY + strong price signal should be CONSIDER, got {risky_result[0]['final_rec']}"
    )
    print("\u2705 RISKY verdict preserved (RISKY + strong buy = CONSIDER, not SKIP)")

    print("\u2705 All deterministic checks passed")

run_deterministic_checks()


✅ extract_total_cost tests passed
✅ Shipping-aware profit calculation verified
✅ Product mismatch override verified (PASS + z=-3.0 = SKIP)
✅ RISKY verdict preserved (RISKY + strong buy = CONSIDER, not SKIP)
✅ All deterministic checks passed
